<div align='center'>

# MoQAdam: Moment-Decoupled Quantized Adam
## Empirical Benchmark — 5 Tasks · Ablation · Memory Profiling · Statistical Tests

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?style=flat-square)](https://python.org) 
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0%2B-orange?style=flat-square)](https://pytorch.org) 
[![GPU](https://img.shields.io/badge/GPU-H100%2FA100-green?style=flat-square)]()

</div>

---

## Abstract

MoQAdam addresses three independent failure modes of naive error-feedback Adam:

| Innovation | Acronym | Failure mode fixed |
|------------|---------|--------------------|
| Variance-Normalized Quantization | **VNQ** | Scale-dependent quantization error |
| SNR-Gated Residual Feedback | **SGRF** | Residual drift in low-SNR (noisy) regime |
| Decoupled Moment Update | **DMU** | Biased second-moment from quantized-gradient feedback |

**Benchmark tasks (5 total for publication-strength evidence):**
1. CIFAR-10 — image classification (ResNet-18)
2. CIFAR-100 — fine-grained image classification (ResNet-18)
3. WikiText-2 — word-level language modelling (2-layer LSTM)
4. Penn Treebank (PTB via HuggingFace) — language modelling (2-layer LSTM)
5. Synthetic ill-conditioned quadratic — controlled optimizer convergence analysis

> All datasets loaded via HuggingFace `datasets` — no direct URL downloads that 503.
> Batch sizes tuned for H100/A100 throughput.


---

## Section 1 — Environment & Configuration

In [1]:
import subprocess, sys
def _pip(pkg):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
for pkg in ['torch','torchvision','datasets','matplotlib','seaborn',
            'pandas','scipy','tqdm','tabulate','Pillow']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: _pip(pkg)
print('All dependencies satisfied.')


c:\Users\Samir Guenchi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All dependencies satisfied.


In [2]:
import os, time, math, csv, warnings, random, copy
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from tabulate import tabulate
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')

# ─── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
seed_everything()

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
OUT_DIR = Path('results'); OUT_DIR.mkdir(exist_ok=True)

# ─── H100/A100 optimised batch sizes ────────────────────────────────────────
# Tune these if you run on smaller GPUs
CIFAR_BATCH  = 512   # was 128 — fully utilises H100 SM throughput
LM_BATCH     = 256   # was 20  — massive speedup on H100
LM_SEQ_LEN   = 70    # was 35  — doubles context per forward pass
WORKERS      = 4

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.15)
plt.rcParams.update({'figure.dpi':130,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

print(f'Device         : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch        : {torch.__version__}')
print(f'CIFAR batch    : {CIFAR_BATCH}  LM batch: {LM_BATCH}  LM seq: {LM_SEQ_LEN}')
print(f'Output dir     : {OUT_DIR.resolve()}')


Device         : cuda
GPU            : NVIDIA GeForce RTX 3050 Ti Laptop GPU
VRAM total     : 4.3 GB
PyTorch        : 2.6.0+cu124
CIFAR batch    : 512  LM batch: 256  LM seq: 70
Output dir     : D:\project\QNN\publish_v1\QodAam\results


---

## Section 2 — MoQAdam Core Implementation

### Mathematical Background

**Standard Adam update:**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\theta_{t+1} = \theta_t - \eta \, \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$$

**Problem with naïve EF-Adam (updates both moments from $q_t$):**
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)q_t^2 \implies \mathbb{E}[\hat{v}_t] \neq \mathbb{E}[g_t^2] \quad (\text{biased})$$

**MoQAdam DMU fix** — $v_t$ from original $g_t$, $m_t$ from recovered $\tilde{g}_t$:
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\mathbf{g_t}^2, \quad \sigma_t = \sqrt{\hat{v}_t+\epsilon}$$

**MoQAdam VNQ** — normalize before quantizing:
$$\tilde{g}_t = g_t/\sigma_t, \quad c_t = \tilde{g}_t + \gamma e_{t-1}, \quad q_t = Q(c_t), \quad e_t = \gamma(c_t - q_t)$$

**MoQAdam SGRF gate:**
$$\gamma_t = \min\!\left(1,\; \frac{1}{\rho}\cdot\overline{\operatorname{SNR}}_t\right), \quad \overline{\operatorname{SNR}}_t = \mathbb{E}\!\left[\hat{m}_{t-1}^2/(\hat{v}_{t-1}+\epsilon)\right]$$


In [3]:
class MoQAdam(torch.optim.Optimizer):
    """
    Moment-Decoupled Quantized Adam.

    Flags
    -----
    quant_enabled    : enable VNQ   (False → raw-space quantization only)
    snr_gate_enabled : enable SGRF  (False → gamma=1 always)
    decouple_moments : enable DMU   (False → v_t from q_t, naive-EF style)
    """
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999), eps=1e-8,
                 weight_decay=1e-4, quant_bits=8, snr_ref=0.10,
                 clip_norm=1.0, residual_dtype=torch.float16,
                 quant_enabled=True, snr_gate_enabled=True, decouple_moments=True):
        super().__init__(params, dict(
            lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
            quant_bits=quant_bits, snr_ref=snr_ref, clip_norm=clip_norm,
            residual_dtype=residual_dtype,
            quant_enabled=quant_enabled, snr_gate_enabled=snr_gate_enabled,
            decouple_moments=decouple_moments))

    @staticmethod
    def _quantize(x: Tensor, bits: int) -> Tuple[Tensor, Tensor]:
        """Uniform symmetric mid-tread quantizer in normalized space."""
        L = 2**(bits-1) - 1
        xm = x.abs().amax().clamp(min=1e-8)
        s  = xm / L
        q  = x.div(s).round_().clamp_(-L, L).mul_(s)
        return q, x - q

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()

        # ── Optional global gradient clipping ───────────────────────────────
        clip = self.param_groups[0].get('clip_norm', 0.0)
        if clip > 0:
            grads = [p.grad for g in self.param_groups
                     for p in g['params'] if p.grad is not None]
            if grads:
                norm = torch.stack([g.float().norm() for g in grads]).norm()
                if norm > clip:
                    scale = clip / (norm + 1e-6)
                    for g in grads: g.mul_(scale)

        for group in self.param_groups:
            lr     = group['lr']
            b1, b2 = group['betas']
            eps    = group['eps']
            wd     = group['weight_decay']
            bits   = group['quant_bits']
            rho    = group['snr_ref']
            rdt    = group['residual_dtype']
            q_on   = group['quant_enabled']
            snr_on = group['snr_gate_enabled']
            dmu_on = group['decouple_moments']

            for p in group['params']:
                if p.grad is None or not torch.isfinite(p.grad).all(): continue
                g = p.grad.float()
                s = self.state[p]

                if not s:
                    s['step'] = 0
                    s['m']    = torch.zeros_like(g)
                    s['v']    = torch.zeros_like(g)
                    s['e']    = torch.zeros_like(g,
                                   dtype=rdt if rdt != torch.float32 else g.dtype)

                s['step'] += 1; t = s['step']
                m, v, e = s['m'], s['v'], s['e']

                if not q_on:
                    # Ablation / disabled: standard AdamW
                    v.mul_(b2).addcmul_(g, g, value=1-b2)
                    m.mul_(b1).add_(g, alpha=1-b1)
                    mh = m/(1-b1**t); vh = v/(1-b2**t)
                    u  = mh/(vh.sqrt().add_(eps))
                    if wd: u.add_(p.data, alpha=wd)
                    p.data.add_(u, alpha=-lr); continue

                # ── DMU: update v from ORIGINAL g ───────────────────────────
                if dmu_on:
                    v.mul_(b2).addcmul_(g, g, value=1-b2)
                vh_bc = v / (1-b2**t)
                sigma = vh_bc.add(eps).sqrt_()

                # ── SGRF: SNR gate ────────────────────────────────────────────
                gamma = 1.0
                if snr_on and t >= 2:
                    bc1p = 1-b1**(t-1); bc2p = 1-b2**(t-1)
                    snr  = (m/bc1p).pow(2).div((v/bc2p).add(eps)).mean()
                    gamma = float(snr.clamp_(0, rho).div_(rho))

                # ── VNQ ──────────────────────────────────────────────────────
                ef = e.float() if e.dtype != torch.float32 else e
                q_norm, r = self._quantize(g/sigma + gamma*ef, bits)
                r.mul_(gamma)
                e.copy_(r.to(e.dtype) if e.dtype != torch.float32 else r)
                q_g = q_norm * sigma   # de-normalized

                if not dmu_on:  # naive: v from quantized gradient
                    v.mul_(b2).addcmul_(q_g, q_g, value=1-b2)
                    vh_bc = v / (1-b2**t)

                m.mul_(b1).add_(q_g, alpha=1-b1)
                mh = m/(1-b1**t)
                u  = mh / (vh_bc.sqrt().add_(eps))
                if wd: u.add_(p.data, alpha=wd)
                p.data.add_(u, alpha=-lr)
        return loss


print('MoQAdam ✓')


MoQAdam ✓


---

## Section 3 — Baseline Optimizers

In [4]:
class NaiveEFAdam(torch.optim.Optimizer):
    """EF in raw gradient space, unconditional residual, biased v_t."""
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999),
                 eps=1e-8, weight_decay=1e-4, quant_bits=8):
        super().__init__(params,dict(lr=lr,betas=betas,eps=eps,
                                     weight_decay=weight_decay,quant_bits=quant_bits))
    @staticmethod
    def _q(x,bits):
        S=2**(bits-1)-1; xm=x.abs().amax().clamp(1e-8); sc=xm/S
        q=x.div(sc).round().clamp(-S,S).mul(sc); return q, x-q
    @torch.no_grad()
    def step(self, closure=None):
        loss=None
        if closure: 
            with torch.enable_grad(): loss=closure()
        for g in self.param_groups:
            lr,(b1,b2),eps,wd,bits=g['lr'],g['betas'],g['eps'],g['weight_decay'],g['quant_bits']
            for p in g['params']:
                if p.grad is None: continue
                grad=p.grad.float(); st=self.state[p]
                if not st: st.update({'step':0,'m':torch.zeros_like(grad),
                                      'v':torch.zeros_like(grad),'e':torch.zeros_like(grad)})
                st['step']+=1; t=st['step']; m,v,e=st['m'],st['v'],st['e']
                q,r=self._q(grad+e,bits); e.copy_(r)
                v.mul_(b2).addcmul_(q,q,value=1-b2)
                m.mul_(b1).add_(q,alpha=1-b1)
                mh=m/(1-b1**t); vh=v/(1-b2**t)
                u=mh/(vh.sqrt().add_(eps))
                if wd: u.add_(p.data,alpha=wd)
                p.data.add_(u,alpha=-lr)
        return loss


class EF21Adam(torch.optim.Optimizer):
    """EF21 top-k contractive compressor + Adam (Richtárik et al. 2021)."""
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999),
                 eps=1e-8, weight_decay=1e-4, quant_bits=8):
        super().__init__(params,dict(lr=lr,betas=betas,eps=eps,
                                     weight_decay=weight_decay,quant_bits=quant_bits))
    @staticmethod
    def _topk(x,bits):
        n=x.numel(); k=max(1,int(n*bits/32))
        f=x.flatten(); _,idx=f.abs().topk(k)
        q=torch.zeros_like(f); q[idx]=f[idx]
        q=q.view_as(x); return q, x-q
    @torch.no_grad()
    def step(self, closure=None):
        loss=None
        if closure:
            with torch.enable_grad(): loss=closure()
        for g in self.param_groups:
            lr,(b1,b2),eps,wd,bits=g['lr'],g['betas'],g['eps'],g['weight_decay'],g['quant_bits']
            for p in g['params']:
                if p.grad is None: continue
                grad=p.grad.float(); st=self.state[p]
                if not st: st.update({'step':0,'m':torch.zeros_like(grad),
                                      'v':torch.zeros_like(grad),'g_prev':grad.clone()})
                st['step']+=1; t=st['step']; m,v=st['m'],st['v']
                c,_=self._topk(grad-st['g_prev'],bits)
                gef=st['g_prev']+c; st['g_prev'].copy_(gef)
                v.mul_(b2).addcmul_(gef,gef,value=1-b2)
                m.mul_(b1).add_(gef,alpha=1-b1)
                mh=m/(1-b1**t); vh=v/(1-b2**t)
                u=mh/(vh.sqrt().add_(eps))
                if wd: u.add_(p.data,alpha=wd)
                p.data.add_(u,alpha=-lr)
        return loss


print('NaiveEFAdam ✓   EF21Adam ✓')


NaiveEFAdam ✓   EF21Adam ✓


---

## Section 4 — Model Architectures

| Model | Task | ~Params |
|-------|------|---------|
| ResNet-18 (CIFAR variant) | CIFAR-10 / CIFAR-100 | 11.2 M |
| 2-layer LSTM | WikiText-2 / PTB LM | ≈20 M |
| Logistic Regression | Synthetic quadratic | 1 M |


In [5]:
def make_resnet18(num_classes=10):
    """ResNet-18 with 3×3 stem/no maxpool — standard CIFAR modification."""
    import torchvision.models as tvm
    m = tvm.resnet18(weights=None, num_classes=num_classes)
    m.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    return m


class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed=256, hidden=512, layers=2, drop=0.5):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed)
        self.drop   = nn.Dropout(drop)
        self.lstm   = nn.LSTM(embed, hidden, layers, dropout=drop, batch_first=False)
        self.proj   = nn.Linear(hidden, vocab_size)
        self.H = hidden; self.L = layers
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        nn.init.zeros_(self.proj.bias)
    def forward(self, x, h=None):
        e = self.drop(self.embed(x))
        o, h = self.lstm(e, h)
        return self.proj(self.drop(o)), h
    def init_hidden(self, bs, dev):
        w = next(self.parameters())
        return (w.new_zeros(self.L,bs,self.H), w.new_zeros(self.L,bs,self.H))


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'ResNet-18 CIFAR-10  : {count_params(make_resnet18(10))/1e6:.2f}M')
print(f'ResNet-18 CIFAR-100 : {count_params(make_resnet18(100))/1e6:.2f}M')
print(f'LSTM LM  (vocab 30k): {count_params(LSTMLanguageModel(30000))/1e6:.2f}M')


ResNet-18 CIFAR-10  : 11.17M
ResNet-18 CIFAR-100 : 11.22M
LSTM LM  (vocab 30k): 26.75M


---

## Section 5 — Optimizer Factory

> All optimizers: `lr=1e-3`, `weight_decay=1e-4`, `betas=(0.9,0.999)` — no per-optimizer tuning.

In [6]:
LR = 1e-3
WD = 1e-4

DISPLAY = {
    'adamw':              'AdamW',
    'naive_ef_8bit':      'Naive EF-Adam (8-bit)',
    'naive_ef_4bit':      'Naive EF-Adam (4-bit)',
    'ef21_8bit':          'EF21-Adam (8-bit)',
    'ef21_4bit':          'EF21-Adam (4-bit)',
    'moqadam_8bit':       'MoQAdam (8-bit)',
    'moqadam_4bit':       'MoQAdam (4-bit)',
    # ablation
    'moqadam_vnq':        '+VNQ only',
    'moqadam_sgrf':       '+SGRF only',
    'moqadam_dmu':        '+DMU only',
    'moqadam_vnq_sgrf':   '+VNQ+SGRF',
    'moqadam_vnq_dmu':    '+VNQ+DMU',
    'moqadam_full':       'MoQAdam (full=8bit)',
}

def make_opt(name, params, bits=8):
    c = dict(lr=LR, weight_decay=WD, betas=(0.9,0.999), eps=1e-8)
    if name == 'adamw': return AdamW(params, **c)
    if 'naive_ef' in name:
        b = 4 if '4' in name else 8
        return NaiveEFAdam(params, **c, quant_bits=b)
    if 'ef21' in name:
        b = 4 if '4' in name else 8
        return EF21Adam(params, **c, quant_bits=b)
    if 'moqadam' in name:
        b    = 4 if '4bit' in name else 8
        vnq  = any(x in name for x in ('vnq','full','8bit','4bit'))
        sgrf = any(x in name for x in ('sgrf','full','8bit','4bit'))
        dmu  = any(x in name for x in ('dmu','full','8bit','4bit'))
        return MoQAdam(params, **c, quant_bits=b,
                       snr_ref=0.10, clip_norm=1.0,
                       residual_dtype=torch.float16,
                       quant_enabled=vnq, snr_gate_enabled=sgrf,
                       decouple_moments=dmu)
    raise ValueError(name)


MAIN_OPTS     = ['adamw','naive_ef_8bit','ef21_8bit','moqadam_8bit','moqadam_4bit']
ABLATION_OPTS = ['adamw','naive_ef_8bit','moqadam_vnq','moqadam_sgrf',
                 'moqadam_dmu','moqadam_vnq_sgrf','moqadam_vnq_dmu','moqadam_full']

print('Optimizer factory ready.')
print('Main opts    :', MAIN_OPTS)
print('Ablation opts:', ABLATION_OPTS)


Optimizer factory ready.
Main opts    : ['adamw', 'naive_ef_8bit', 'ef21_8bit', 'moqadam_8bit', 'moqadam_4bit']
Ablation opts: ['adamw', 'naive_ef_8bit', 'moqadam_vnq', 'moqadam_sgrf', 'moqadam_dmu', 'moqadam_vnq_sgrf', 'moqadam_vnq_dmu', 'moqadam_full']


---

## Section 6 — Data Loading

All vision datasets loaded via **HuggingFace `datasets`** — no direct CS-Toronto URL that 503s.
LM datasets also via HuggingFace.


In [7]:
# ─── CIFAR via HuggingFace (avoids cs.toronto.edu 503) ──────────────────────
import torchvision.transforms as T
from torch.utils.data import Dataset as TDataset

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

class HFCifarDataset(TDataset):
    """Wraps a HuggingFace CIFAR split with torchvision transforms."""
    def __init__(self, hf_split, transform=None):
        self.data      = hf_split
        self.transform = transform
        # HF cifar10/cifar100 use 'img' and 'label' / 'fine_label'
        self.label_col = 'fine_label' if 'fine_label' in hf_split.features else 'label'
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img  = item['img']
        lbl  = item[self.label_col]
        if self.transform: img = self.transform(img)
        return img, lbl


def get_cifar_loaders(num_classes=10, batch_size=CIFAR_BATCH):
    from datasets import load_dataset
    name = 'cifar100' if num_classes == 100 else 'cifar10'
    print(f'Downloading {name} via HuggingFace …')
    ds = load_dataset(name, trust_remote_code=True)

    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.4, 0.4, 0.4),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

    train_ds = HFCifarDataset(ds['train'], train_tf)
    test_ds  = HFCifarDataset(ds['test'],  test_tf)
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                           num_workers=WORKERS, pin_memory=True, drop_last=True)
    te_loader = DataLoader(test_ds,  batch_size=512, shuffle=False,
                           num_workers=WORKERS, pin_memory=True)
    print(f'  {name}: train={len(train_ds):,}  test={len(test_ds):,}')
    return tr_loader, te_loader


# ─── LM data ─────────────────────────────────────────────────────────────────
def get_lm_loaders(dataset_name='wikitext-2', batch_size=LM_BATCH, seq_len=LM_SEQ_LEN):
    from datasets import load_dataset
    if dataset_name == 'ptb':
        # Penn Treebank via HuggingFace
        raw = load_dataset('ptb_text_only', trust_remote_code=True)
        splits = ('train','validation','test')
        text_col = 'sentence'
    else:
        raw = load_dataset('wikitext','wikitext-2-raw-v1')
        splits = ('train','validation','test')
        text_col = 'text'

    vocab: Dict[str,int] = {'<eos>':0,'<unk>':1}
    def tokenize(txt):
        return [vocab.setdefault(w, len(vocab))
                for w in txt.replace('\n','<eos>').split()]

    def batchify(split):
        toks = []
        for row in raw[split][text_col]: toks.extend(tokenize(row))
        data = torch.tensor(toks, dtype=torch.long)
        n = data.size(0) // batch_size
        return data[:n*batch_size].view(batch_size,-1).t().contiguous()

    train_d = batchify('train'); val_d = batchify('validation')
    V = len(vocab)

    def get_batch(src, i):
        l = min(seq_len, src.size(0)-1-i)
        return src[i:i+l], src[i+1:i+1+l].reshape(-1)

    print(f'  {dataset_name}: train={train_d.numel():,} tok  '
          f'val={val_d.numel():,} tok  vocab={V:,}  batch={batch_size}  seq={seq_len}')
    return train_d, val_d, V, get_batch


---

## Section 7 — Training & Evaluation Helpers

In [8]:
def train_cifar_epoch(model, loader, opt, device, scaler=None):
    model.train(); criterion = nn.CrossEntropyLoss()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        if scaler:
            with torch.autocast(device.type, dtype=torch.bfloat16):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        else:
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def eval_cifar(model, loader, device):
    model.eval(); criterion = nn.CrossEntropyLoss()
    tl, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        tl      += criterion(logits, y).item() * x.size(0)
        correct += logits.argmax(1).eq(y).sum().item()
        total   += x.size(0)
    return tl/total, 100.0*correct/total


def train_lm_epoch(model, data, opt, device, get_batch, seq_len=LM_SEQ_LEN, clip=0.25):
    model.train(); criterion = nn.CrossEntropyLoss()
    total = 0.0; h = model.init_hidden(data.size(1), device)
    nb = (data.size(0)-1) // seq_len
    for i in range(0, data.size(0)-1, seq_len):
        d, t = get_batch(data, i)
        d, t = d.to(device), t.to(device)
        h = tuple(x.detach() for x in h)
        opt.zero_grad()
        out, h = model(d, h)
        loss = criterion(out.view(-1, out.size(2)), t)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        total += loss.item()
    return total / nb


@torch.no_grad()
def eval_lm(model, data, device, get_batch, seq_len=LM_SEQ_LEN):
    model.eval(); criterion = nn.CrossEntropyLoss()
    total = 0.0; h = model.init_hidden(data.size(1), device)
    for i in range(0, data.size(0)-1, seq_len):
        d, t = get_batch(data, i)
        d, t = d.to(device), t.to(device)
        h = tuple(x.detach() for x in h)
        out, h = model(d, h)
        total += criterion(out.view(-1,out.size(2)), t).item() * d.size(0)
    return math.exp(total / (data.size(0)-1))


print('Training helpers ✓')


Training helpers ✓


---

## Section 8 — Generic Benchmark Runner

In [9]:
def run_vision_benchmark(task_name, num_classes, epochs, opts=MAIN_OPTS):
    """
    Full CIFAR-10 or CIFAR-100 benchmark.
    Returns (history_dict, final_dict).
    """
    try:
        tr_loader, te_loader = get_cifar_loaders(num_classes)
    except Exception as e:
        print(f'[ERROR] {task_name} data: {e}'); return {}, {}

    history: Dict[str, Dict] = {}; final: Dict[str, Dict] = {}
    print(f'\n{task_name} ({epochs} epochs, batch={CIFAR_BATCH}, device={DEVICE})')
    print('='*65)

    for opt_name in opts:
        seed_everything()
        model  = make_resnet18(num_classes).to(DEVICE)
        # torch.compile if available (2× speedup on H100)
        if hasattr(torch, 'compile') and DEVICE.type == 'cuda':
            try: model = torch.compile(model, mode='reduce-overhead')
            except: pass
        opt    = make_opt(opt_name, model.parameters())
        sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

        hist  = {'train_loss':[],'val_loss':[],'val_acc':[],'ep_time':[]}
        best  = 0.0; t0 = time.time()
        for ep in range(1, epochs+1):
            te = time.time()
            tl = train_cifar_epoch(model, tr_loader, opt, DEVICE, scaler)
            vl, acc = eval_cifar(model, te_loader, DEVICE)
            sched.step()
            ep_t = time.time()-te
            hist['train_loss'].append(tl); hist['val_loss'].append(vl)
            hist['val_acc'].append(acc);   hist['ep_time'].append(ep_t)
            if acc > best: best = acc
            if ep % 20 == 0 or ep == epochs:
                print(f'  [{opt_name:20s}] ep{ep:3d}  acc={acc:.2f}%  '
                      f'best={best:.2f}%  ep_time={ep_t:.1f}s  '
                      f'total={time.time()-t0:.0f}s')

        _, fa = eval_cifar(model, te_loader, DEVICE)
        history[opt_name] = hist
        final[opt_name]   = {'top1_acc':fa, 'best_acc':best,
                              'total_min':(time.time()-t0)/60,
                              'avg_ep_s': float(np.mean(hist['ep_time']))}
        print(f'  [{opt_name:20s}] FINAL acc={fa:.2f}%  best={best:.2f}%')

    tag = task_name.lower().replace('-','').replace(' ','')
    pd.DataFrame(final).T.to_csv(OUT_DIR/f'{tag}_results.csv')
    print(f'Saved: {OUT_DIR}/{tag}_results.csv')
    return history, final


def run_lm_benchmark(task_name, dataset_name, epochs, opts=MAIN_OPTS):
    """
    Full LM benchmark on any HuggingFace text dataset.
    Returns (history_dict, final_dict).
    """
    try:
        tr, val, V, get_batch = get_lm_loaders(dataset_name)
        tr = tr.to(DEVICE); val = val.to(DEVICE)
    except Exception as e:
        print(f'[ERROR] {task_name} data: {e}'); return {}, {}

    history: Dict[str, Dict] = {}; final: Dict[str, Dict] = {}
    print(f'\n{task_name} LM ({epochs} epochs, batch={LM_BATCH}, device={DEVICE})')
    print('='*65)

    for opt_name in opts:
        seed_everything()
        model = LSTMLanguageModel(V).to(DEVICE)
        opt   = make_opt(opt_name, model.parameters())
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        hist = {'train_loss':[],'val_ppl':[],'ep_time':[]}
        best = float('inf'); t0 = time.time()
        for ep in range(1, epochs+1):
            te = time.time()
            tl  = train_lm_epoch(model, tr, opt, DEVICE, get_batch)
            ppl = eval_lm(model, val, DEVICE, get_batch)
            sched.step()
            ep_t = time.time()-te
            hist['train_loss'].append(tl); hist['val_ppl'].append(ppl)
            hist['ep_time'].append(ep_t)
            if ppl < best: best = ppl
            if ep % 10 == 0 or ep == epochs:
                print(f'  [{opt_name:20s}] ep{ep:2d}  ppl={ppl:.2f}  '
                      f'best={best:.2f}  ep_time={ep_t:.1f}s  '
                      f'total={time.time()-t0:.0f}s')

        history[opt_name] = hist
        final[opt_name]   = {'final_ppl':ppl,'best_ppl':best,
                              'total_min':(time.time()-t0)/60}
        print(f'  [{opt_name:20s}] FINAL ppl={ppl:.2f}  best={best:.2f}')

    tag = task_name.lower().replace('-','').replace(' ','')
    pd.DataFrame(final).T.to_csv(OUT_DIR/f'{tag}_results.csv')
    print(f'Saved: {OUT_DIR}/{tag}_results.csv')
    return history, final


print('Benchmark runners ✓')


Benchmark runners ✓


---

## Section 9 — Benchmark Configuration

Adjust epochs below. The full-paper protocol is shown; quick values for H100 smoke-testing.

| Task | Full | Quick (H100 ~1h total) |
|------|------|------------------------|
| CIFAR-10 | 100 ep | 50 ep |
| CIFAR-100 | 100 ep | 50 ep |
| WikiText-2 LM | 50 ep | 30 ep |
| PTB LM | 50 ep | 30 ep |
| Synthetic | N/A | instant |


In [10]:
# ── Edit these ──────────────────────────────────────────────────────────────
C10_EPOCHS  = 100   # CIFAR-10 epochs
C100_EPOCHS = 100   # CIFAR-100 epochs
WT2_EPOCHS  =  50   # WikiText-2 epochs
PTB_EPOCHS  =  50   # PTB epochs
ABL_EPOCHS  =  30   # Ablation epochs (CIFAR-100, 4-bit)
# ────────────────────────────────────────────────────────────────────────────

print('Configuration:')
for name, val in [('CIFAR-10',C10_EPOCHS),('CIFAR-100',C100_EPOCHS),
                   ('WikiText-2',WT2_EPOCHS),('PTB',PTB_EPOCHS),
                   ('Ablation',ABL_EPOCHS)]:
    print(f'  {name:15s}: {val} epochs')


Configuration:
  CIFAR-10       : 100 epochs
  CIFAR-100      : 100 epochs
  WikiText-2     : 50 epochs
  PTB            : 50 epochs
  Ablation       : 30 epochs


---

## Section 10 — Task 1: CIFAR-10 Classification (ResNet-18)

**CIFAR-10** is a 10-class image benchmark. Including it alongside CIFAR-100 lets us:
- Show optimizer behaviour at different class granularity
- Verify MoQAdam doesn't degrade on simpler tasks
- Get results faster (fewer classes = quicker convergence)


In [11]:
c10_history, c10_final = run_vision_benchmark('CIFAR-10', 10, C10_EPOCHS)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cifar10' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating test split: 100%|██████████| 10000/10000 [00:00<00:00, 42811.86 examples/s]


  cifar10: train=50,000  test=10,000

CIFAR-10 (100 epochs, batch=512, device=cuda)


RuntimeError: DataLoader worker (pid(s) 18116, 14280, 18092, 19800) exited unexpectedly

---

## Section 11 — Task 2: CIFAR-100 Classification (ResNet-18)

**CIFAR-100** has 100 fine-grained classes — much harder than CIFAR-10.
The tighter per-class margin stresses optimizer precision, making quantization effects more visible.


In [ ]:
c100_history, c100_final = run_vision_benchmark('CIFAR-100', 100, C100_EPOCHS)


---

## Section 12 — Task 3: WikiText-2 Language Modelling

WikiText-2 is a standard NLP benchmark (2M train tokens).  
We use `batch_size=256` and `seq_len=70` (tunable in Section 9) to fully utilise H100 throughput.
**Note:** increased batch size = ~10–20× faster than the original `batch_size=20`.


In [ ]:
wt2_history, wt2_final = run_lm_benchmark('WikiText-2','wikitext-2', WT2_EPOCHS)


---

## Section 13 — Task 4: Penn Treebank (PTB) Language Modelling

PTB is the canonical word-level LM benchmark. Its smaller vocabulary (~10k) and
smaller corpus (~1M tokens) test optimizers under lower data-diversity conditions
compared to WikiText-2.


In [ ]:
ptb_history, ptb_final = run_lm_benchmark('PTB','ptb', PTB_EPOCHS)


---

## Section 14 — Task 5: Synthetic Ill-Conditioned Quadratic

**Why a synthetic benchmark?**

Real tasks conflate optimizer quality with model capacity and data complexity.
A controlled quadratic lets us isolate optimizer behavior:
- Fixed objective, known optimum
- Tunable condition number κ (we use κ=100, typical of ill-conditioned NNs)
- Simulated quantization noise at controlled SNR
- Measures convergence speed and final suboptimality

**Objective:** $\min_{\theta} \frac{1}{2}\theta^\top A\theta + b^\top\theta$  
where $A = \text{diag}(1, \kappa/(d-1), 2\kappa/(d-1), \ldots, \kappa)$, condition number $\kappa=100$.


In [ ]:
def run_synthetic_benchmark(d=10_000, kappa=100, n_steps=2000,
                             bits=8, noise_std=0.01, opts=MAIN_OPTS):
    """
    Minimize f(θ) = 0.5 θᵀ A θ + bᵀθ  subject to quantized gradient updates.
    A = diag(eigs)  with eigs linearly spaced from 1 to kappa.
    Ground truth optimum: θ* = -A⁻¹ b.
    """
    torch.manual_seed(SEED)
    eigs = torch.linspace(1.0, float(kappa), d, device=DEVICE)
    A    = eigs
    b    = torch.randn(d, device=DEVICE) * 0.1
    theta_star = -b / A   # analytic optimum
    f_star     = 0.5*(theta_star*A*theta_star).sum() + (b*theta_star).sum()

    print(f'\nSynthetic quadratic (d={d:,}, κ={kappa}, bits={bits}, steps={n_steps})')
    print(f'  f* = {f_star.item():.6f}')
    print('='*65)

    results = {}
    for opt_name in opts:
        torch.manual_seed(SEED)
        theta = nn.Parameter(torch.randn(d, device=DEVICE) * 0.01)
        opt   = make_opt(opt_name, [theta], bits=bits)

        losses = []
        for step in range(n_steps):
            opt.zero_grad()
            # Exact gradient: A*θ + b, plus small observation noise
            g_exact = A * theta.data + b
            noise   = torch.randn_like(g_exact) * noise_std
            theta.grad = (g_exact + noise).detach().float()
            opt.step()
            with torch.no_grad():
                f = (0.5*(theta*A*theta).sum() + (b*theta).sum()).item()
            losses.append(f)

        final_subopt = abs(losses[-1] - f_star.item())
        print(f'  [{opt_name:20s}]  final f={losses[-1]:.6f}  '
              f'suboptimality={final_subopt:.2e}')
        results[opt_name] = {'losses': losses, 'final_f': losses[-1],
                              'suboptimality': final_subopt}
    return results


synth_results = run_synthetic_benchmark()


In [ ]:
# ── Plot synthetic convergence ────────────────────────────────────────────────
if synth_results:
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, (k, v) in enumerate(synth_results.items()):
        c = colors[i]
        label = DISPLAY.get(k, k)
        axes[0].plot(v['losses'], color=c, lw=1.5, label=label)
        axes[1].semilogy(v['losses'], color=c, lw=1.5, label=label)

    for ax, t in zip(axes, ['Linear scale','Log scale (convergence rate)']):
        ax.set(title=f'Synthetic Quadratic — {t}',
               xlabel='Step', ylabel='f(θ)')
        ax.legend(fontsize=8)

    plt.tight_layout()
    fig.savefig(OUT_DIR/'synthetic_convergence.pdf', bbox_inches='tight')
    plt.show()
    print('Saved: synthetic_convergence.pdf')

    # Summary table
    rows = [(DISPLAY.get(k,k), v['final_f'], v['suboptimality'])
            for k,v in synth_results.items()]
    print(tabulate(rows,
                   headers=['Optimizer','Final f(θ)','Sub-optimality'],
                   tablefmt='github', floatfmt='.4e'))


---

## Section 15 — Convergence Visualisations (All Tasks)

In [ ]:
def plot_convergence(histories_list, titles, metrics, ylabels, filename):
    """Generic multi-task convergence plot."""
    n = len(histories_list)
    if n == 0: print('No data'); return
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1: axes = [axes]
    for ax, hist, title, metric, ylabel in zip(axes,histories_list,titles,metrics,ylabels):
        for i,(k,h) in enumerate(hist.items()):
            ax.plot(h[metric], color=colors[i], lw=1.5, label=DISPLAY.get(k,k))
        ax.set(title=title, xlabel='Epoch', ylabel=ylabel)
        ax.legend(fontsize=8)
    plt.suptitle('Convergence Across All Tasks', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUT_DIR/filename, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filename}')


# Vision accuracy curves
vision_hists = [(h,t) for h,t in [(c10_history,'CIFAR-10'),(c100_history,'CIFAR-100')] if h]
if vision_hists:
    plot_convergence([h for h,_ in vision_hists],
                     [t for _,t in vision_hists],
                     ['val_acc']*len(vision_hists),
                     ['Top-1 Acc (%)']*len(vision_hists),
                     'vision_convergence.pdf')

# LM perplexity curves
lm_hists = [(h,t) for h,t in [(wt2_history,'WikiText-2'),(ptb_history,'PTB')] if h]
if lm_hists:
    plot_convergence([h for h,_ in lm_hists],
                     [t for _,t in lm_hists],
                     ['val_ppl']*len(lm_hists),
                     ['Perplexity']*len(lm_hists),
                     'lm_convergence.pdf')


---

## Section 16 — Ablation Study (CIFAR-100, 4-bit)

Progressive component activation isolates each innovation's individual contribution.


In [ ]:
ablation_history, ablation_final = run_vision_benchmark(
    'Ablation-CIFAR100', 100, ABL_EPOCHS, opts=ABLATION_OPTS
)


In [ ]:
if ablation_final:
    VNQ_SET  = {'moqadam_vnq','moqadam_vnq_sgrf','moqadam_vnq_dmu','moqadam_8bit','moqadam_4bit','moqadam_full'}
    SGRF_SET = {'moqadam_sgrf','moqadam_vnq_sgrf','moqadam_8bit','moqadam_4bit','moqadam_full'}
    DMU_SET  = {'moqadam_dmu','moqadam_vnq_dmu','moqadam_8bit','moqadam_4bit','moqadam_full'}

    labels = [DISPLAY.get(k,k) for k in ablation_final]
    accs   = [ablation_final[k]['best_acc'] for k in ablation_final]
    colors = plt.cm.Paired.colors[:len(labels)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Horizontal bar chart
    bars = ax1.barh(labels, accs, color=colors, alpha=0.85)
    baseline = list(ablation_final.values())[0]['best_acc']
    ax1.axvline(baseline, ls='--', color='gray', lw=1.5, label='AdamW baseline')
    for bar, v in zip(bars, accs):
        ax1.text(v+0.05, bar.get_y()+bar.get_height()/2,
                 f'{v:.2f}%', va='center', fontsize=9)
    ax1.set(xlabel='Best Top-1 Accuracy (%)',
            title='Ablation: Each Component\'s Contribution')
    ax1.legend(fontsize=9)

    # Delta vs AdamW
    deltas = [a - baseline for a in accs]
    bar_colors = ['green' if d > 0 else 'red' for d in deltas]
    ax2.barh(labels, deltas, color=bar_colors, alpha=0.75)
    ax2.axvline(0, color='black', lw=1)
    ax2.set(xlabel='Δ Accuracy vs AdamW (pp)',
            title='Ablation: Gain / Loss over AdamW Baseline')

    plt.tight_layout()
    fig.savefig(OUT_DIR/'ablation_study.pdf', bbox_inches='tight')
    plt.show()
    print('Saved: ablation_study.pdf')
else:
    print('[SKIPPED] No ablation results.')


---

## Section 17 — Memory Footprint Analysis

| Component | AdamW | Naive EF-Adam | MoQAdam (fp16 residual) |
|-----------|-------|---------------|-------------------------|
| m_t | 1P fp32 | 1P fp32 | 1P fp32 |
| v_t | 1P fp32 | 1P fp32 | 1P fp32 |
| e_t | — | 1P fp32 | 0.5P fp16 |
| **Total** | **2P** | **3P** | **2.5P** |


In [ ]:
def measure_mem(opt_name, bits=8, n=1_000_000):
    model = nn.Linear(n, 1, bias=False).to(DEVICE)
    opt   = make_opt(opt_name, model.parameters(), bits=bits)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    x = torch.randn(4, n, device=DEVICE)
    loss = model(x).sum(); loss.backward(); opt.step(); opt.zero_grad()
    state_b = sum(v.element_size()*v.numel()
                  for p in model.parameters() for v in opt.state[p].values()
                  if isinstance(v, torch.Tensor))
    del model, opt
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return {'state_MB': state_b/1e6, 'B_per_param': state_b/n}


MEM_CONFIGS = [('adamw',8),('naive_ef_8bit',8),('naive_ef_4bit',4),
               ('ef21_8bit',8),('moqadam_8bit',8),('moqadam_4bit',4)]
mem_res = {}
print('Memory profiling (1M parameter model)\n')
for opt_name, bits in MEM_CONFIGS:
    r = measure_mem(opt_name, bits)
    mem_res[opt_name] = r
    print(f'  {opt_name:22s}  {r["state_MB"]:6.2f} MB  '
          f'({r["B_per_param"]:.1f} B/param)')

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ks = list(mem_res.keys()); vals = [mem_res[k]['state_MB'] for k in ks]
bars = ax.bar([DISPLAY.get(k,k) for k in ks], vals,
              color=plt.cm.tab10.colors[:len(ks)], alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{v:.1f}', ha='center', fontsize=9)
ax.set(ylabel='State Memory (MB)', title='Optimizer State Memory — 1M Parameters')
plt.xticks(rotation=20, ha='right', fontsize=9)
plt.tight_layout()
fig.savefig(OUT_DIR/'memory.pdf', bbox_inches='tight'); plt.show()


---

## Section 18 — Statistical Significance Testing

Paired two-sided Wilcoxon signed-rank test on the **last 20% of training epochs**
(more robust to non-normality than t-test; preferred for small-sample optimizer comparisons).

H₀: MoQAdam (8-bit) and baseline have equal median epoch-wise accuracy.
H₁: MoQAdam (8-bit) achieves systematically higher accuracy.


In [ ]:
def wilcoxon_test(a_hist, b_hist, tail=None):
    n = len(a_hist)
    if tail is None: tail = max(1, n//5)
    a = np.array(a_hist[-tail:])
    b = np.array(b_hist[-tail:])
    if np.allclose(a, b): return 0.0, 1.0
    w, p = stats.wilcoxon(a, b)
    return float(w), float(p)


def print_significance(histories, task_name, metric, ref_key='moqadam_8bit'):
    if not histories or ref_key not in histories:
        print(f'[{task_name}] No data or ref key missing. Skipping.'); return
    ref = histories[ref_key][metric]
    print(f'\n{task_name} — Wilcoxon test vs {DISPLAY.get(ref_key,ref_key)}')
    print(f'{"Optimizer":<30s}  {"W":>8s}  {"p-value":>10s}  Result')
    print('-'*70)
    for k, h in histories.items():
        if k == ref_key: continue
        w, p = wilcoxon_test(ref, h[metric])
        sig  = '** p<0.01' if p<0.01 else ('* p<0.05' if p<0.05 else 'n.s.')
        direction = '↑ better' if np.mean(ref[-len(ref)//5:]) > np.mean(h[metric][-len(h[metric])//5:]) else '↓ worse'
        print(f'{DISPLAY.get(k,k):<30s}  {w:>8.1f}  {p:>10.4f}  {sig} {direction}')


print_significance(c10_history,   'CIFAR-10',   'val_acc')
print_significance(c100_history,  'CIFAR-100',  'val_acc')
print_significance(wt2_history,   'WikiText-2', 'val_ppl')
print_significance(ptb_history,   'PTB',        'val_ppl')


---

## Section 19 — Auto-Generated LaTeX Tables

Paste into your paper. Check `results/paper_tables.tex` for the full file.


In [ ]:
def fmt(x): return f'{x:.2f}' if isinstance(x,(float,int)) else '--'

DISPLAY_TEX = {
    'adamw':         'AdamW',
    'naive_ef_8bit': 'Naive EF-Adam (8-bit)',
    'naive_ef_4bit': 'Naive EF-Adam (4-bit)',
    'ef21_8bit':     r'EF21-Adam~\cite{richtarik2021ef21} (8-bit)',
    'ef21_4bit':     r'EF21-Adam~\cite{richtarik2021ef21} (4-bit)',
    'moqadam_8bit':  r'\textbf{MoQAdam} (8-bit)',
    'moqadam_4bit':  r'\textbf{MoQAdam} (4-bit)',
    'moqadam_vnq':   '+VNQ only',
    'moqadam_sgrf':  '+SGRF only',
    'moqadam_dmu':   '+DMU only',
    'moqadam_vnq_sgrf':'+VNQ+SGRF',
    'moqadam_vnq_dmu': '+VNQ+DMU',
    'moqadam_full':  r'\textbf{MoQAdam} (full)',
}
VNQ_S  = {'moqadam_vnq','moqadam_vnq_sgrf','moqadam_vnq_dmu','moqadam_8bit','moqadam_4bit','moqadam_full'}
SGRF_S = {'moqadam_sgrf','moqadam_vnq_sgrf','moqadam_8bit','moqadam_4bit','moqadam_full'}
DMU_S  = {'moqadam_dmu','moqadam_vnq_dmu','moqadam_8bit','moqadam_4bit','moqadam_full'}
ck = lambda k, S: r'\cmark' if k in S else r'\xmark'


def table_vision(results, caption, label):
    if not results: return '% [no data]\n'
    L = [r'\begin{table}[t]', r'\centering',
         f'\\caption{{{caption}}}', f'\\label{{{label}}}',
         r'\begin{tabular}{lccc}', r'\toprule',
         r'\textbf{Optimizer} & \textbf{Bits} & \textbf{Top-1 (\%)} & \textbf{Best (\%)} \\',
         r'\midrule']
    for k in ['adamw','naive_ef_8bit','naive_ef_4bit','ef21_8bit','ef21_4bit',
               'moqadam_8bit','moqadam_4bit']:
        if k not in results: continue
        bits = '8' if '8bit' in k else ('4' if '4bit' in k else 'N/A')
        L.append(f'{DISPLAY_TEX.get(k,k)} & {bits} & '
                 f'{fmt(results[k]["top1_acc"])} & {fmt(results[k]["best_acc"])} \\\\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}', '']
    return '\n'.join(L)


def table_lm(results, caption, label):
    if not results: return '% [no data]\n'
    L = [r'\begin{table}[t]', r'\centering',
         f'\\caption{{{caption}}}', f'\\label{{{label}}}',
         r'\begin{tabular}{lcc}', r'\toprule',
         r'\textbf{Optimizer} & \textbf{Final PPL} & \textbf{Best PPL} \\',
         r'\midrule']
    for k in ['adamw','naive_ef_8bit','ef21_8bit','moqadam_8bit','moqadam_4bit']:
        if k not in results: continue
        L.append(f'{DISPLAY_TEX.get(k,k)} & {fmt(results[k]["final_ppl"])} & '
                 f'{fmt(results[k]["best_ppl"])} \\\\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}', '']
    return '\n'.join(L)


def table_ablation(results):
    if not results: return '% [no data]\n'
    L = [r'\begin{table}[t]', r'\centering',
         r'\caption{Ablation study on CIFAR-100 (4-bit, 30 epochs). Each component evaluated independently.}',
         r'\label{tab:ablation}',
         r'\begin{tabular}{lcccc}', r'\toprule',
         r'\textbf{Config} & \textbf{VNQ} & \textbf{SGRF} & \textbf{DMU} & \textbf{Best Acc (\%)} \\',
         r'\midrule']
    order = ['adamw','naive_ef_8bit','moqadam_vnq','moqadam_sgrf',
             'moqadam_dmu','moqadam_vnq_sgrf','moqadam_vnq_dmu','moqadam_full']
    for k in order:
        if k not in results: continue
        acc = fmt(results[k].get('best_acc', results[k].get('top1_acc','--')))
        L.append(f'{DISPLAY_TEX.get(k,k)} & {ck(k,VNQ_S)} & {ck(k,SGRF_S)} & '
                 f'{ck(k,DMU_S)} & {acc} \\\\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(L)


def table_memory(mem_res):
    if not mem_res: return '% [no data]\n'
    L = [r'\begin{table}[t]', r'\centering',
         r'\caption{Optimizer state memory (1M parameters). MoQAdam fp16 residual saves 16.7\% vs Naive EF-Adam.}',
         r'\label{tab:memory}',
         r'\begin{tabular}{lcc}', r'\toprule',
         r'\textbf{Optimizer} & \textbf{State (MB)} & \textbf{B / param} \\',
         r'\midrule']
    for k, v in mem_res.items():
        L.append(f'{DISPLAY_TEX.get(k,k)} & {v["state_MB"]:.2f} & {v["B_per_param"]:.1f} \\\\')
    L += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(L)


latex_out = '\n\n'.join([
    '% ─── Table 1: CIFAR-10 ───────────────────────────────────────────────',
    table_vision(c10_final,
        'Top-1 accuracy (\\%) on CIFAR-10 (ResNet-18, 100 epochs, seed=42).',
        'tab:cifar10'),
    '% ─── Table 2: CIFAR-100 ──────────────────────────────────────────────',
    table_vision(c100_final,
        'Top-1 accuracy (\\%) on CIFAR-100 (ResNet-18, 100 epochs, seed=42).',
        'tab:cifar100'),
    '% ─── Table 3: WikiText-2 ─────────────────────────────────────────────',
    table_lm(wt2_final,
        'Perplexity on WikiText-2 validation (2-layer LSTM, 50 epochs). Lower is better.',
        'tab:wt2'),
    '% ─── Table 4: PTB ────────────────────────────────────────────────────',
    table_lm(ptb_final,
        'Perplexity on Penn Treebank validation (2-layer LSTM, 50 epochs). Lower is better.',
        'tab:ptb'),
    '% ─── Table 5: Ablation ───────────────────────────────────────────────',
    table_ablation(ablation_final),
    '% ─── Table 6: Memory ────────────────────────────────────────────────',
    table_memory(mem_res),
])

(OUT_DIR/'paper_tables.tex').write_text(latex_out)
print('Saved: results/paper_tables.tex\n')
print(latex_out)


---

## Section 20 — Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)
colors = plt.cm.tab10.colors

# Row 0: CIFAR-10 acc curve | CIFAR-100 acc curve | Vision bar chart
for col, (hist, title) in enumerate([(c10_history,'CIFAR-10'),(c100_history,'CIFAR-100')]):
    ax = fig.add_subplot(gs[0, col])
    if hist:
        for i,(k,h) in enumerate(hist.items()):
            ax.plot(h['val_acc'],color=colors[i],lw=1.5,label=DISPLAY.get(k,k))
    ax.set(title=f'{title} Val Acc',xlabel='Epoch',ylabel='Acc (%)')
    ax.legend(fontsize=7, ncol=1)

ax_vbar = fig.add_subplot(gs[0, 2])
all_final = {**c10_final, **c100_final}
if all_final:
    ks2 = list(set(c10_final)&set(c100_final)) if c10_final and c100_final else []
    if ks2:
        x = np.arange(len(ks2)); w=0.35
        ax_vbar.bar(x-w/2,[c10_final[k]['best_acc'] for k in ks2],w,label='CIFAR-10',alpha=0.8)
        ax_vbar.bar(x+w/2,[c100_final[k]['best_acc'] for k in ks2],w,label='CIFAR-100',alpha=0.8)
        ax_vbar.set_xticks(x); ax_vbar.set_xticklabels([DISPLAY.get(k,k) for k in ks2],
                                                         rotation=30,ha='right',fontsize=7)
        ax_vbar.set(ylabel='Best Acc (%)',title='Vision: Best Accuracy')
        ax_vbar.legend(fontsize=8)

# Row 1: WikiText-2 | PTB | LM bar
for col, (hist, title) in enumerate([(wt2_history,'WikiText-2'),(ptb_history,'PTB')]):
    ax = fig.add_subplot(gs[1, col])
    if hist:
        for i,(k,h) in enumerate(hist.items()):
            ax.plot(h['val_ppl'],color=colors[i],lw=1.5,label=DISPLAY.get(k,k))
    ax.set(title=f'{title} Perplexity',xlabel='Epoch',ylabel='PPL')
    ax.legend(fontsize=7)

ax_lbar = fig.add_subplot(gs[1, 2])
for h_lm, label in [(wt2_final,'WT2'),(ptb_final,'PTB')]:
    if h_lm:
        ks_lm = list(h_lm.keys())
        ax_lbar.plot([DISPLAY.get(k,k) for k in ks_lm],
                     [h_lm[k]['best_ppl'] for k in ks_lm],'o-',label=label,lw=1.5)
ax_lbar.set(title='LM Best PPL',ylabel='PPL')
ax_lbar.tick_params(axis='x',rotation=30)
ax_lbar.legend(fontsize=8)

# Row 2: Synthetic | Ablation | Memory
ax_syn = fig.add_subplot(gs[2, 0])
if synth_results:
    for i,(k,v) in enumerate(synth_results.items()):
        ax_syn.semilogy(v['losses'],color=colors[i],lw=1.3,label=DISPLAY.get(k,k))
ax_syn.set(title='Synthetic Quadratic (log)',xlabel='Step',ylabel='f(θ)')
ax_syn.legend(fontsize=7)

ax_abl = fig.add_subplot(gs[2, 1])
if ablation_final:
    ks_a = list(ablation_final.keys())
    ax_abl.barh([DISPLAY.get(k,k) for k in ks_a],
                [ablation_final[k]['best_acc'] for k in ks_a],
                color=colors[:len(ks_a)],alpha=0.8)
ax_abl.set(title='Ablation Best Acc',xlabel='Acc (%)')

ax_mem = fig.add_subplot(gs[2, 2])
if mem_res:
    ks_m = list(mem_res.keys())
    ax_mem.bar([DISPLAY.get(k,k) for k in ks_m],
               [mem_res[k]['state_MB'] for k in ks_m],
               color=colors[:len(ks_m)],alpha=0.8)
    ax_mem.tick_params(axis='x',rotation=20)
ax_mem.set(title='Memory (MB/1M params)',ylabel='MB')

fig.suptitle('MoQAdam Benchmark — Full Summary (5 Tasks)',fontsize=15,fontweight='bold')
fig.savefig(OUT_DIR/'summary_dashboard.pdf',bbox_inches='tight')
plt.show()
print('Saved: summary_dashboard.pdf')


---

## Section 21 — Output Inventory & Reproducibility

In [ ]:
print('='*65)
print('All output files')
print('='*65)
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name:<40s}  {f.stat().st_size:>10,} bytes')

print(f'\nSeed: {SEED} | LR: {LR} | WD: {WD}')
print(f'CIFAR batch: {CIFAR_BATCH} | LM batch: {LM_BATCH} | LM seq: {LM_SEQ_LEN}')
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')
